In [1]:
from service_connector import ServiceConnector
import asyncio
import nest_asyncio


nest_asyncio.apply()
quantile_values = [0.75, 0.85, 0.95, 0.99, 0.999]
service_connector = ServiceConnector()
loop = asyncio.get_event_loop()
github_data = loop.run_until_complete(service_connector.get_data("2024-01-01", "2024-04-01", 1000))
github_data

,repo,pushes,avg_push_size,pull_requests,merged_pull_requests_ratio,issues,closed_issues_ratio,watches,forks,language,license_name,is_deleted_or_private
0,brueckna2020/MDFit,12,1.250000,2,0.0,0,0.000000,18,0,Python,MIT License,False
1,zdsjjtTLG/TrackIt,69,1.014493,0,0.0,2,0.000000,11,4,Python,MIT License,False
2,sherlock-vaib/omg-bro,1,1.000000,0,0.0,0,0.000000,32,0,None,None,True
3,chaojie/ComfyUI-Pymunk,1,1.000000,0,0.0,0,0.000000,14,1,Python,None,False
4,rato4785/rato47851,0,0.000000,0,0.0,0,0.000000,27,0,None,None,True
...,...,...,...,...,...,...,...,...,...,...,...,...
994,benedictus79/Hotm4rtei,42,1.738095,40,0.0,0,0.000000,17,0,Python,None,False
995,r-three/phatgoose,6,1.166667,0,0.0,2,0.000000,29,2,Python,MIT License,False
996,ubuygold/go-noss,6,1.500000,1,0.0,47,0.234043,200,129,Go,None,False
997,ahdqbflyjd/Adobe-Illustrator,1,1.000000,0,0.0,0,0.000000,26,0,None,None,True


In [2]:

quantile_data = service_connector.get_quantiles("2024-01-01",
                             "2024-02-01", 
                             ['forks', 'pushes','avg_push_size', 
                              'pull_requests', 'merged_pull_requests_ratio', 
                              'issues', 'closed_issues_ratio', 'watches'],
                             quantile_values)
quantile_data

,forks,pushes,avg_push_size,pull_requests,merged_pull_requests_ratio,issues,closed_issues_ratio,watches
0.750,2.000,6.000,1.113752,7.000,0.000000,4.000,0.500000,2.000
0.850,2.000,11.000,1.668593,12.000,0.133333,8.000,0.666667,3.000
0.950,6.000,31.000,4.666667,36.000,0.500000,24.000,1.000000,10.000
0.990,24.090,115.090,112.001412,101.000,0.625000,84.000,1.000000,52.000
0.999,143.809,625.034,3334.000000,325.798,1.000000,388.079,1.000000,417.405


In [3]:
from github_data_converter import GithubDataConverter
from pattern_miner import PatternMiner


github_data_converter = GithubDataConverter()
transactions = github_data_converter.convert_data_to_transactions(github_data, quantile_data)

pattern_miner = PatternMiner()
patterns = pattern_miner.mine_patterns(transactions, 0.1, 0.1, 1)
patterns

d:\WorkData\SusuRepos\final-qualifying-work\project\github_data_converter.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  clean_data.drop('repo', axis=1, inplace=True)


,antecedents,consequents,support,confidence,lift
0,(Мало пушей),(Удален или скрыт),0.4087,0.5525,1.3099
1,(Удален или скрыт),(Мало пушей),0.4087,0.9689,1.3099
2,(Удален или скрыт),(Мало размера пушей),0.4218,1.0000,1.1073
3,(Мало размера пушей),(Удален или скрыт),0.4218,0.4670,1.1073
4,(Удален или скрыт),(Мало запросов на слияние),0.4218,1.0000,1.0421
...,...,...,...,...,...
2903,"(Мало закрытых ишью), (Мало форков), (Выше сре...",(Мало ишью),0.6145,0.9636,1.0235
2904,"(Мало закрытых ишью), (Мало ишью), (Мало форков)",(Выше среднего звезд),0.6145,0.7454,1.0583
2905,"(Выше среднего звезд), (Мало ишью), (Мало форков)",(Мало закрытых ишью),0.6145,0.9967,1.0018
2916,"(Мало закрытых ишью), (Много звезд), (Мало фор...",(Мало ишью),0.1877,0.9538,1.0131


In [4]:
patterns.to_excel("data/patterns.xlsx")